## Create Related Work Generation Datatset from Multi-XScience

In [ ]:
import gzip
import json
from pathlib import Path

import pandas as pd

DATA_DIR = Path("***************************************/Multi-XScience/data")
RANDOM_STATE = 42
N_PER_BIN = 25


def load_json_gz(path, split_name):
    with gzip.open(path, "rt", encoding="utf-8") as f:
        data = json.load(f)

    rows = []

    # If file is dict format
    if isinstance(data, dict):
        for row_id, row in data.items():
            row = dict(row)
            row["row_id"] = row_id
            row["split"] = split_name
            rows.append(row)

    # If file is list format
    elif isinstance(data, list):
        for i, row in enumerate(data):
            row = dict(row)
            row["row_id"] = i
            row["split"] = split_name
            rows.append(row)

    else:
        raise ValueError(f"Unexpected format in {path}")

    return rows


def count_raw_refs(ref_abstract):
    if isinstance(ref_abstract, dict):
        return len(ref_abstract)
    return 0


# -------------------------------
# Load and combine all splits
# -------------------------------

all_rows = []

for split in ["train", "val", "test"]:
    file_path = DATA_DIR / f"{split}.json.gz"
    split_rows = load_json_gz(file_path, split)
    all_rows.extend(split_rows)

df = pd.DataFrame(all_rows)

print("Total rows:", len(df))
print(df["split"].value_counts())


# -------------------------------
# Count references
# -------------------------------

df["raw_ref_count"] = df["ref_abstract"].apply(count_raw_refs)

print("\nRaw reference count summary:")
print(df["raw_ref_count"].describe())


# -------------------------------
# Create reference-count bins
# -------------------------------

df["ref_bin"] = pd.cut(
    df["raw_ref_count"],
    bins=[0, 5, 10, 15, 20],
    labels=["1_5_refs", "6_10_refs", "11_15_refs", "16_20_refs"],
    include_lowest=True
)

print("\nRows per reference bin:")
print(df["ref_bin"].value_counts().sort_index())


# -------------------------------
# Stratified sampling: 25 per bin
# -------------------------------

subset = (
    df.groupby("ref_bin", group_keys=False, observed=True)
      .apply(lambda x: x.sample(
          n=min(len(x), N_PER_BIN),
          random_state=RANDOM_STATE
      ))
      .reset_index(drop=True)
)

print("\nFinal subset size:", len(subset))
print(subset["ref_bin"].value_counts().sort_index())
print("\nOriginal split distribution in sampled subset:")
print(subset["split"].value_counts())


# -------------------------------
# Save subset
# -------------------------------

subset.to_csv("multixscience_100_ref_stratified_subset.csv", index=False)

# JSONL to preserve nested fields like ref_abstract properly
subset.to_json("multixscience_100_ref_stratified_subset.jsonl", orient="records", lines=True, force_ascii=False)

print("\nSaved:")
print("multixscience_100_ref_stratified_subset.csv")
print("multixscience_100_ref_stratified_subset.jsonl") # renamed to "related_work_generation_dataset.jsonl" in github repository

Total rows: 40528
split
train    30369
test      5093
val       5066
Name: count, dtype: int64

Raw reference count summary:
count    40528.000000
mean         4.084732
std          3.430126
min          1.000000
25%          1.000000
50%          3.000000
75%          6.000000
max         20.000000
Name: raw_ref_count, dtype: float64

Rows per reference bin:
ref_bin
1_5_refs      30392
6_10_refs      7631
11_15_refs     1990
16_20_refs      515
Name: count, dtype: int64

Final subset size: 100
ref_bin
1_5_refs      25
6_10_refs     25
11_15_refs    25
16_20_refs    25
Name: count, dtype: int64

Original split distribution in sampled subset:
split
train    80
val      10
test     10
Name: count, dtype: int64

Saved:
multixscience_100_ref_stratified_subset.csv
multixscience_100_ref_stratified_subset.jsonl


/tmp/ipykernel_2948107/2157892546.py:94: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(


In [7]:
subset

,aid,mid,abstract,related_work,ref_abstract,row_id,split,raw_ref_count,ref_bin
0,1605.08143,2951649892,Though deliberation is a critical component of...,The problem of scaling up decision-making is a...,"{'@cite_30': {'mid': '', 'abstract': ''}, '@ci...",20320,train,4,1_5_refs
1,1906.11373,2954325026,Analysis of player tracking data for American ...,Since the NFL tracking data was released early...,"{'@cite_2': {'mid': '2197859468', 'abstract': ...",3225,train,1,1_5_refs
2,1709.00588,2752284099,Batched sparse (BATS) code is a promising tech...,Ng @cite_22 studied the performance of finite-...,"{'@cite_22': {'mid': '2949775276', 'abstract':...",11435,train,5,1_5_refs
3,1610.07238,2952702088,"In visual tracking, part-based trackers are at...",Our localization process is inspired by @cite_...,"{'@cite_0': {'mid': '1980475298', 'abstract': ...",18906,train,3,1_5_refs
4,1709.06389,2759134725,We propose a new framework for the recognition...,"In the method proposed in this work, instead a...","{'@cite_9': {'mid': '1978799108', 'abstract': ...",11123,train,1,1_5_refs
...,...,...,...,...,...,...,...,...,...
95,1903.12476,2931083803,Recent progress on salient object detection ma...,Salient object detection is a very active rese...,"{'@cite_47': {'mid': '2953227099', 'abstract':...",13771,train,19,16_20_refs
96,1904.06505,2618902759,Objective assessment of image quality is funda...,"From the feature extraction point of view, thr...","{'@cite_35': {'mid': '2132984323', 'abstract':...",13222,train,17,16_20_refs
97,1611.08209,2950450215,We consider the problem of fault-tolerant para...,"In several studies, when the environment is no...","{'@cite_30': {'mid': '2126572568', 'abstract':...",18412,train,16,16_20_refs
98,1904.07396,2940805846,Deep convolutional neural networks perform bet...,"In this section, we present and discuss recent...","{'@cite_61': {'mid': '2122911643', 'abstract':...",13179,train,18,16_20_refs
